In [ ]:
import base64
import csv
import json
import os
import re
import tempfile
import traceback
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path
from typing import Any, Optional
from urllib.parse import quote, urljoin

import cv2
import numpy as np
import requests
from bs4 import BeautifulSoup
from IPython.display import Image, display
from ultralytics import YOLO


In [ ]:
SOURCE_PAGE_URL = "https://www.fortcollins.gov/Activities/Parks-Natural-Areas-and-Trails/Natural-Areas/Visit/Parking-Lot-Cameras"
SENSERA_NODE_ID = "M78031286A82"
SNAPSHOT_URL = f"https://public.senserasystems.com/public/embed/{SENSERA_NODE_ID}"
LATEST_PIC_URL = f"https://public.senserasystems.com/public/latestPic/{SENSERA_NODE_ID}"
MEDIA_BASE_URL = f"https://public.senserasystems.com/public/media/{SENSERA_NODE_ID}/"
MODEL_PATH = os.getenv("MODEL_PATH", "yolov8m.pt")
VEHICLE_CLASSES = [2, 3, 5, 7]
CONFIDENCE = float(os.getenv("CONFIDENCE", "0.45"))
IOU = float(os.getenv("IOU", "0.45"))
IMAGE_SIZE = int(os.getenv("IMAGE_SIZE", "1280"))
USER_AGENT = os.getenv(
    "USER_AGENT",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
)
GITHUB_REPOSITORY = os.getenv("GITHUB_REPOSITORY", "VolpeUSDOT/Public-Lands-Computer-Vision")
GITHUB_BRANCH = os.getenv("GITHUB_BRANCH", "main")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "WCT").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "Ft. Collins"
ANNOTATED_IMAGE_OUTPUT_PATH = OUTPUT_DIR / "gateway_latest_annotated.jpg"
FEED_OUTPUT_PATH = OUTPUT_DIR / "gateway_latest_feed.txt"
JSON_OUTPUT_PATH = OUTPUT_DIR / "gateway_latest_feed.json"
GITHUB_BASE_PATH = "IP3/Ft. Collins"
GITHUB_ANNOTATED_IMAGE_PATH = f"{GITHUB_BASE_PATH}/gateway_latest_annotated.jpg"
GITHUB_FEED_PATH = f"{GITHUB_BASE_PATH}/gateway_latest_feed.txt"
GITHUB_JSON_PATH = f"{GITHUB_BASE_PATH}/gateway_latest_feed.json"


@dataclass
class DetectionBox:
    class_id: int
    class_name: str
    confidence: float
    xyxy: list[float]


@dataclass
class RunResult:
    status: str
    timestamp_utc: str
    source_page_url: str
    image_url: Optional[str] = None
    model_path: str = MODEL_PATH
    vehicle_count: int = 0
    detections: list[DetectionBox] = field(default_factory=list)
    message: Optional[str] = None
    error: Optional[str] = None


def build_headers(user_agent: str) -> dict[str, str]:
    return {"User-Agent": user_agent}


def fetch_bytes(url: str, headers: dict[str, str], timeout: int = 15) -> requests.Response:
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response


def download_image(image_url: str, headers: dict[str, str]) -> np.ndarray:
    response = fetch_bytes(image_url, headers=headers)
    if not response.content:
        raise ValueError(f"Downloaded empty image content from {image_url}")

    content_type = response.headers.get("Content-Type", "").lower()
    if content_type and not content_type.startswith("image/"):
        body_preview = response.text.strip().replace("\n", " ")[:200]
        raise ValueError(
            f"Resolved URL did not return an image (content-type={content_type!r}): {image_url}; body={body_preview!r}"
        )

    if response.content.lstrip().startswith((b"<", b"<!doctype", b"<?xml")):
        body_preview = response.text.strip().replace("\n", " ")[:200]
        raise ValueError(f"Resolved URL returned HTML/XML instead of an image: {image_url}; body={body_preview!r}")

    image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"OpenCV could not decode the downloaded image from {image_url}")
    return img


def extract_candidate_urls(page_text: str, base_url: str) -> list[str]:
    soup = BeautifulSoup(page_text, "html.parser")
    candidates: list[str] = []

    for tag in soup.find_all(["img", "source", "iframe", "a"]):
        for attr in ("src", "data-src", "href"):
            value = tag.get(attr)
            if value:
                candidates.append(urljoin(base_url, value))

    patterns = [
        r'https?://[^"\'\s>]+(?:jpg|jpeg|png|gif)',
        r'https?://[^"\'\s>]+snapshot[^"\'\s>]*',
        r'https?://[^"\'\s>]+public/embed/[^"\'\s>]+',
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, page_text, re.IGNORECASE):
            candidates.append(urljoin(base_url, match.group(0)))

    unique_candidates: list[str] = []
    seen: set[str] = set()
    for candidate in candidates:
        if candidate and candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)
    return unique_candidates


def resolve_sensera_latest_image(headers: dict[str, str]) -> tuple[np.ndarray, str]:
    filename = fetch_bytes(LATEST_PIC_URL, headers=headers).text.strip()
    if not filename:
        raise ValueError("Sensera latestPic endpoint returned an empty filename")
    return download_image(urljoin(MEDIA_BASE_URL, filename), headers=headers), urljoin(MEDIA_BASE_URL, filename)


def resolve_image_url(candidate_urls: list[str], headers: dict[str, str]) -> tuple[np.ndarray, str]:
    queue = [url for url in candidate_urls if url]
    seen: set[str] = set()

    while queue:
        url = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)

        response = fetch_bytes(url, headers=headers)
        content_type = response.headers.get("Content-Type", "").lower()

        try:
            image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
            img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError(f"OpenCV could not decode the downloaded image from {url}")
            return img, url
        except Exception:
            pass

        if content_type.startswith("text/") or response.text.lstrip().startswith("<"):
            queue.extend(extract_candidate_urls(response.text, url))

    raise ValueError("Could not resolve a usable image URL")


@lru_cache(maxsize=1)
def load_model(model_path: str) -> YOLO:
    return YOLO(model_path)


def serialize_detections(result: Any) -> list[DetectionBox]:
    if not result.boxes:
        return []

    names = result.names or {}
    detections: list[DetectionBox] = []
    for box in result.boxes:
        class_id = int(box.cls[0]) if getattr(box, "cls", None) is not None else -1
        confidence = float(box.conf[0]) if getattr(box, "conf", None) is not None else 0.0
        coords = [float(value) for value in box.xyxy[0].tolist()]
        detections.append(
            DetectionBox(
                class_id=class_id,
                class_name=str(names.get(class_id, class_id)),
                confidence=confidence,
                xyxy=coords,
            )
        )
    return detections


def annotate_image(img: np.ndarray, results: Any) -> np.ndarray:
    if not results:
        return img
    return results[0].plot()


def result_to_dict(result: RunResult) -> dict[str, Any]:
    payload = asdict(result)
    payload["detections"] = [asdict(detection) for detection in result.detections]
    payload["output_schema"] = "gateway_vehicle_count_v1"
    return payload


def result_to_json_text(result: RunResult) -> str:
    return json.dumps(result_to_dict(result), indent=2) + "\n"


def result_to_feed_text(result: RunResult) -> str:
    lines = [
        "Fort Collins gateway vehicle feed",
        f"status: {result.status}",
        f"timestamp_utc: {result.timestamp_utc}",
        f"source_page_url: {result.source_page_url}",
        f"image_url: {result.image_url or ''}",
        f"model_path: {result.model_path}",
        f"vehicle_count: {result.vehicle_count}",
    ]

    if result.message:
        lines.append(f"message: {result.message}")
    if result.error:
        lines.append(f"error: {result.error}")

    if result.detections:
        lines.append("detections:")
        for detection in result.detections:
            lines.append(
                f"- {detection.class_name} (class_id={detection.class_id}, confidence={detection.confidence:.3f}) xyxy={detection.xyxy}"
            )
    else:
        lines.append("detections: none")

    lines.append("output_schema: gateway_vehicle_count_v1")
    return "\n".join(lines) + "\n"


def resolve_writable_output_path(output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        output_path.touch(exist_ok=True)
        return output_path
    except (PermissionError, OSError):
        fallback_path = Path(tempfile.gettempdir()) / output_path.name
        fallback_path.parent.mkdir(parents=True, exist_ok=True)
        return fallback_path


def write_text(output_path: Path, content: str) -> Path:
    writable_path = resolve_writable_output_path(output_path)
    writable_path.write_text(content, encoding="utf-8")
    return writable_path


def write_annotated_image(img: np.ndarray, output_path: Path) -> Path:
    writable_path = resolve_writable_output_path(output_path)
    if cv2.imwrite(str(writable_path), img):
        return writable_path
    raise ValueError(f"Failed to write annotated image to {output_path}")


def github_contents_api_url(repository: str, file_path: str) -> str:
    return f"https://api.github.com/repos/{repository}/contents/{quote(file_path.lstrip('/'))}"


def get_github_headers(token: str) -> dict[str, str]:
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }


def publish_text_to_github(content_text: str, repository: str, branch: str, file_path: str, token: str, message: str) -> tuple[bool, str]:
    headers = get_github_headers(token)
    url = github_contents_api_url(repository, file_path)
    current_sha: Optional[str] = None

    response = requests.get(url, headers=headers, params={"ref": branch}, timeout=10)
    if response.status_code == 200:
        payload = response.json()
        current_sha = payload.get("sha")
        existing_content = payload.get("content")
        encoding = payload.get("encoding")
        if existing_content and encoding == "base64":
            decoded = base64.b64decode(existing_content).decode("utf-8")
            if decoded == content_text:
                return True, f"GitHub file already up to date: {file_path}"
    elif response.status_code != 404:
        response.raise_for_status()

    body: dict[str, Any] = {
        "message": message,
        "content": base64.b64encode(content_text.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }
    if current_sha:
        body["sha"] = current_sha

    response = requests.put(url, headers=headers, json=body, timeout=20)
    response.raise_for_status()
    return True, f"Published {repository}/{file_path} on {branch}"


def publish_image_to_github(image: np.ndarray, repository: str, branch: str, file_path: str, token: str, message: str) -> tuple[bool, str]:
    suffix = Path(file_path).suffix.lower()
    ext = ".png" if suffix == ".png" else ".jpg"
    ok, encoded = cv2.imencode(ext, image)
    if not ok:
        return False, f"Failed to encode annotated image for {file_path}"

    content_text = base64.b64encode(encoded.tobytes()).decode("ascii")
    headers = get_github_headers(token)
    url = github_contents_api_url(repository, file_path)
    current_sha: Optional[str] = None

    response = requests.get(url, headers=headers, params={"ref": branch}, timeout=10)
    if response.status_code == 200:
        payload = response.json()
        current_sha = payload.get("sha")
    elif response.status_code != 404:
        response.raise_for_status()

    body: dict[str, Any] = {
        "message": message,
        "content": content_text,
        "branch": branch,
    }
    if current_sha:
        body["sha"] = current_sha

    response = requests.put(url, headers=headers, json=body, timeout=20)
    response.raise_for_status()
    return True, f"Published {repository}/{file_path} on {branch}"


In [ ]:
headers = build_headers(USER_AGENT)
timestamp_utc = datetime.now(timezone.utc).isoformat()
result: RunResult
annotated_image: np.ndarray | None

try:
    try:
        image, image_url = resolve_sensera_latest_image(headers)
    except Exception:
        image, image_url = resolve_image_url([SNAPSHOT_URL, SOURCE_PAGE_URL], headers)
    model = load_model(MODEL_PATH)
    results = model.predict(
        image,
        classes=VEHICLE_CLASSES,
        conf=CONFIDENCE,
        iou=IOU,
        imgsz=IMAGE_SIZE,
        verbose=False,
    )
    detections = serialize_detections(results[0]) if results else []
    annotated_image = annotate_image(image, results)
    result = RunResult(
        status="ok",
        timestamp_utc=timestamp_utc,
        source_page_url=SOURCE_PAGE_URL,
        image_url=image_url,
        model_path=MODEL_PATH,
        vehicle_count=len(detections),
        detections=detections,
        message=f"Detected {len(detections)} vehicles anywhere in the frame",
    )
except Exception as exc:
    result = RunResult(
        status="error",
        timestamp_utc=timestamp_utc,
        source_page_url=SOURCE_PAGE_URL,
        model_path=MODEL_PATH,
        error=f"{exc}\n{traceback.format_exc()}",
    )
    annotated_image = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_path = write_text(JSON_OUTPUT_PATH, result_to_json_text(result))
feed_path = write_text(FEED_OUTPUT_PATH, result_to_feed_text(result))
image_path = None
if annotated_image is not None:
    image_path = write_annotated_image(annotated_image, ANNOTATED_IMAGE_OUTPUT_PATH)

if GITHUB_TOKEN:
    try:
        publish_text_to_github(result_to_feed_text(result), GITHUB_REPOSITORY, GITHUB_BRANCH, GITHUB_FEED_PATH, GITHUB_TOKEN, "Update gateway feed")
        publish_text_to_github(result_to_json_text(result), GITHUB_REPOSITORY, GITHUB_BRANCH, GITHUB_JSON_PATH, GITHUB_TOKEN, "Update gateway JSON")
        if image_path is not None and annotated_image is not None:
            publish_image_to_github(annotated_image, GITHUB_REPOSITORY, GITHUB_BRANCH, GITHUB_ANNOTATED_IMAGE_PATH, GITHUB_TOKEN, "Update gateway annotated image")
    except Exception as exc:
        print(f"GitHub publish skipped: {exc}")

print(f"Wrote feed to {feed_path}")
print(f"Wrote JSON to {json_path}")
if image_path is not None:
    print(f"Wrote annotated image to {image_path}")
    display(Image(filename=str(image_path)))
print(result_to_feed_text(result))
